## Data Generation

### Objective
This notebook documents how synthetic procurement approval workflow data was generated for the project **"Diagnosing Bottlenecks in Procurement Approval Processes."**

Since real enterprise procurement workflow data was not available, a synthetic dataset was created to simulate sequential approval behavior, waiting delays, processing time variation, rework loops, SLA pressure, and stage-level bottlenecks.

The generated data supports:
- exploratory analysis
- bottleneck diagnosis
- delay prediction
- dashboard-based decision support

In [1]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd().resolve().parents[0]
RAW_DIR = BASE_DIR / "data" / "raw"

stage_df = pd.read_csv(RAW_DIR / "stage_level_raw.csv")
request_df = pd.read_csv(RAW_DIR / "request_level_raw.csv")

print("Stage-level shape:", stage_df.shape)
print("Request-level shape:", request_df.shape)

Stage-level shape: (28472, 17)
Request-level shape: (5000, 22)


### Why Synthetic Data Was Used

Real procurement workflow data is often difficult to access because it may contain:
- confidential vendor information
- internal approval patterns
- operational risk signals
- sensitive organizational process records

To ensure the project could still be executed in a realistic and data-driven way, synthetic data was generated using controlled workflow assumptions.

### Workflow Design Assumptions

The synthetic workflow was designed around a sequential procurement approval process with the following stages:

1. Manager Approval  
2. Finance Approval  
3. Procurement Review  
4. Final Approval  

The generated records were designed to reflect:
- different request types
- different priority levels
- internal vs external vendors
- varying request amounts
- complexity-driven delay behavior
- waiting time before stages
- occasional rework loops
- SLA-based delay classification

In [2]:
stage_df.head()

,Request_ID,Stage,Stage_Order,Role,Department_Stage,Department_Requesting,Request_Type,Priority,Vendor_Type,Request_Amount,Complexity_Score,SLA_Hours,System_Load,Start_Time,End_Time,Processing_Time,Waiting_Time
0,R00001,Manager Approval,1,Manager,Management,Operations,IT Purchase,Low,Internal,122958,3,191.4,0.950714,2025-06-29 15:04:12,2025-06-30 09:00:00,16.45,6.07
1,R00001,Rework to Manager Approval,1,Manager,Management,Operations,IT Purchase,Low,Internal,122958,3,191.4,0.950714,2025-06-30 13:00:00,2025-06-30 13:00:00,0.00,4.00
2,R00001,Manager Approval,1,Manager,Management,Operations,IT Purchase,Low,Internal,122958,3,191.4,0.950714,2025-06-30 15:01:12,2025-07-01 09:00:00,16.86,2.02
3,R00001,Finance Approval,2,Finance Officer,Finance,Operations,IT Purchase,Low,Internal,122958,3,191.4,0.950714,2025-07-01 14:03:36,2025-07-03 13:54:00,47.84,5.06
4,R00001,Rework to Manager Approval,2,Finance Officer,Finance,Operations,IT Purchase,Low,Internal,122958,3,191.4,0.950714,2025-07-03 17:54:00,2025-07-03 17:54:00,0.00,4.00


In [3]:
request_df.head()

,Request_ID,Request_Start,Request_End,Total_Processing,Total_Waiting,SLA_Hours,Request_Type,Priority,Department_Requesting,Vendor_Type,...,System_Load,Num_Stages,Total_TAT,Delay_Ratio,SLA_Breach_Hours,Delayed_Flag,Is_High_Value_Request,Is_High_Complexity,Bottleneck_Stage,Max_Stage_Delay
0,R00001,2025-06-29 15:04:12,2025-07-10 09:00:00,172.34,40.88,191.4,IT Purchase,Low,Operations,Internal,...,0.950714,9,257.93,1.347597,66.53,1,0,0,Finance Approval,58.42
1,R00002,2025-01-08 11:30:00,2025-01-12 09:00:00,48.96,10.62,140.4,IT Purchase,High,Admin,Internal,...,0.034389,4,93.50,0.665954,0.00,0,0,0,Procurement Review,25.54
2,R00003,2025-07-07 10:52:48,2025-07-12 17:58:12,98.21,14.60,217.8,Equipment,Low,IT,Internal,...,0.542696,4,127.09,0.583517,0.00,0,0,0,Finance Approval,51.92
3,R00004,2025-07-12 14:00:36,2025-07-18 09:00:00,89.70,17.95,145.2,Office Supplies,Low,Admin,Internal,...,0.382927,4,138.99,0.957231,0.00,0,0,0,Procurement Review,40.89
4,R00005,2025-03-12 11:22:12,2025-03-17 15:14:24,86.16,22.06,102.6,Office Supplies,High,Admin,External,...,0.289751,7,123.87,1.207310,21.27,1,0,0,Finance Approval,24.26


### Stage-Level Dataset

The stage-level dataset captures each individual workflow stage completed by a request.

Important fields include:
- `Request_ID`
- `Stage`
- `Stage_Order`
- `Start_Time`
- `End_Time`
- `Processing_Time`
- `Waiting_Time`
- `SLA_Hours`
- `System_Load`

This dataset is useful for analyzing where delays occur inside the workflow.

### Request-Level Dataset

The request-level dataset aggregates stage-level activity into one row per request.

Important fields include:
- `Total_TAT`
- `Total_Processing`
- `Total_Waiting`
- `Delay_Ratio`
- `SLA_Breach_Hours`
- `Delayed_Flag`
- `Complexity_Score`
- `Num_Stages`
- `Max_Stage_Delay`
- `Bottleneck_Stage`

This dataset is useful for delay prediction and high-level business analysis.

In [4]:
stage_df[["Stage", "Processing_Time", "Waiting_Time"]].describe()

,Processing_Time,Waiting_Time
count,28472.000000,28472.000000
mean,19.788577,3.548844
std,15.020221,1.537605
min,0.000000,1.120000
25%,8.800000,2.070000
50%,15.940000,3.670000
75%,29.120000,4.500000
max,109.290000,9.000000


In [6]:
request_df[
    [
        "Total_TAT",
        "Total_Processing",
        "Total_Waiting",
        "SLA_Hours",
        "Delay_Ratio",
        "SLA_Breach_Hours",
    ]
].describe()

,Total_TAT,Total_Processing,Total_Waiting,SLA_Hours,Delay_Ratio,SLA_Breach_Hours
count,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000
mean,168.569434,112.684074,20.208540,187.861440,0.939150,23.556542
std,76.380161,53.526071,10.717453,50.350965,0.441598,49.708545
min,46.770000,33.440000,3.640000,102.600000,0.320413,0.000000
25%,116.920000,75.790000,12.610000,145.200000,0.617908,0.000000
50%,141.305000,97.510000,16.710000,191.400000,0.814842,0.000000
75%,197.800000,133.452500,25.120000,232.200000,1.108852,19.790000
max,550.300000,406.410000,69.320000,283.800000,3.982137,373.960000


### Key Generated Features

Several analytical features were built into the synthetic data to support later analysis:

- **Complexity_Score**: represents approval complexity based on request type and amount
- **System_Load**: simulates operational pressure affecting waiting times
- **Delayed_Flag**: identifies whether total turnaround time exceeded SLA
- **Bottleneck_Stage**: captures the stage with the highest stage-level delay
- **Is_High_Value_Request**: flags expensive requests
- **Is_High_Complexity**: flags complex requests

In [7]:
request_df["Delayed_Flag"].value_counts(normalize=True) * 100

Delayed_Flag
0    68.48
1    31.52
Name: proportion, dtype: float64

### Initial Observation

The generated dataset contains both delayed and non-delayed requests, which makes it suitable for:
- bottleneck analysis
- comparative workflow insights
- predictive modeling without extreme target imbalance

This balance is important because a highly one-sided target would make delay prediction unrealistic and analytically weak.

### Conclusion

This notebook established the synthetic data foundation for the project.

The generated workflow data reflects realistic approval dependencies and controlled delay behavior, creating a strong base for:
- exploratory data analysis
- bottleneck diagnosis
- predictive risk modeling
- dashboard-based interpretation